# SmartCloset AI Pipeline Notebook（関数化版）

PoCで検証した処理を、FastAPI化しやすい関数群として整理したNotebookです。

処理の中心は以下です。

- `segment_item()`：YOLOv8-segで衣服を切り出す
- `save_yolo_outputs()`：mask / transparent / annotated を保存する
- `extract_metadata_with_openai()`：背景透過PNGをOpenAIに渡して属性抽出する
- `process_one_image()`：1画像ぶんのAIパイプラインを実行する
- `run_pipeline()`：複数画像を一括処理してCSV保存する


## 1. ライブラリ読み込み

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import json
import os
from dotenv import load_dotenv
import base64
from openai import OpenAI

pd.set_option("display.max_columns", None)


## 2. パス・クラス・設定

Notebookを `SmartCloset_AI/ai_prototype/pipe-line/` で実行する想定です。必要に応じてパスを調整してください。

In [ ]:
BASE_DIR = Path(".")

# 単発開発用画像
INPUT_DIR = BASE_DIR / "input"

# PoC画像を使う場合
TEST_IMAGE_DIR = BASE_DIR / "../PoC/test_images"

# ai_prototype/pipe-line/ から見たモデルパス
MODEL_PATH = Path("../../models/fashionpedia_9class_with_data_augmentation.pt")

CLASS_NAMES = [
    "outer",
    "tops",
    "bottoms",
    "dress",
    "shoes",
    "bag",
    "hat",
    "watch",
    "glasses",
]

IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".webp"]

OUTPUT_DIR = BASE_DIR / "output"
MASK_DIR = OUTPUT_DIR / "masks"
TRANSPARENT_DIR = OUTPUT_DIR / "transparent"
ANNOTATED_DIR = OUTPUT_DIR / "annotated"

for d in [OUTPUT_DIR, MASK_DIR, TRANSPARENT_DIR, ANNOTATED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RESULT_CSV_PATH = OUTPUT_DIR / "pipeline_results.csv"

CONF_THRES = 0.25
OPENAI_MODEL = "gpt-5.4-nano"

print("INPUT_DIR:", INPUT_DIR.resolve())
print("TEST_IMAGE_DIR:", TEST_IMAGE_DIR.resolve())
print("MODEL_PATH:", MODEL_PATH.resolve())
print("model exists:", MODEL_PATH.exists())


## 3. モデル・OpenAIクライアント初期化

`.env` に `OPENAI_API_KEY=...` を設定してください。OpenAIを使わずYOLOだけ確認する場合は、後続で `run_openai=False` にします。

In [ ]:
model_yolo = YOLO(MODEL_PATH)
print("YOLO model loaded")

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    print("OpenAI client initialized")
else:
    client = None
    print("OPENAI_API_KEY not found. OpenAI extraction will be skipped or return error.")


## 4. 入力画像一覧を作成する関数

In [ ]:
def collect_images_from_class_dirs(root_dir, class_names=CLASS_NAMES, image_exts=IMAGE_EXTS):
    """
    classごとのディレクトリから画像一覧DataFrameを作る。

    root_dir/
        outer/
        tops/
        bottoms/
        ...
    """
    root_dir = Path(root_dir)
    image_records = []

    for true_class in class_names:
        class_dir = root_dir / true_class

        if not class_dir.exists():
            print(f"Warning: {class_dir} が存在しません")
            continue

        for img_path in sorted(class_dir.iterdir()):
            if img_path.suffix.lower() in image_exts:
                image_records.append({
                    "image_path": str(img_path),
                    "true_class": true_class,
                    "file_name": img_path.name,
                })

    return pd.DataFrame(image_records)


def collect_images_from_flat_dir(input_dir, true_class=None, image_exts=IMAGE_EXTS):
    """
    1つのディレクトリ直下にある画像一覧DataFrameを作る。
    sample画像などの単発確認用。
    """
    input_dir = Path(input_dir)
    image_records = []

    if not input_dir.exists():
        print(f"Warning: {input_dir} が存在しません")
        return pd.DataFrame(columns=["image_path", "true_class", "file_name"])

    for img_path in sorted(input_dir.iterdir()):
        if img_path.suffix.lower() in image_exts:
            image_records.append({
                "image_path": str(img_path),
                "true_class": true_class,
                "file_name": img_path.name,
            })

    return pd.DataFrame(image_records)


## 5. YOLOセグメンテーション関数

靴のように複数インスタンスが出る場合に備え、最も信頼度が高い代表クラスと同じクラスのマスクを合成します。

In [ ]:
def segment_item(image_path, conf=CONF_THRES):
    """
    画像から対象物をセグメンテーションし、背景透過画像・マスク・YOLO結果・メタ情報を返す。

    Returns:
        rgba: 背景透過RGBA画像 ndarray。失敗時 None
        mask: 0-255のマスク画像 ndarray。失敗時 None
        result: YOLO result object。画像読み込み失敗時 None
        info: YOLO検出情報 dict。失敗時 None
        status: success / image_read_error / no_mask
    """
    image_path = Path(image_path)

    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        return None, None, None, None, "image_read_error"

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    results = model_yolo.predict(
        source=str(image_path),
        conf=conf,
        save=False,
        verbose=False,
    )

    result = results[0]

    if result.masks is None or len(result.masks.data) == 0:
        return None, None, result, None, "no_mask"

    masks = result.masks.data.cpu().numpy()
    boxes = result.boxes

    confs = boxes.conf.cpu().numpy()
    cls_ids = boxes.cls.cpu().numpy().astype(int)

    best_idx = int(np.argmax(confs))
    pred_class_id = int(cls_ids[best_idx])
    pred_class = model_yolo.names[pred_class_id]
    confidence = float(confs[best_idx])

    target_indices = np.where(cls_ids == pred_class_id)[0]

    combined_mask = np.zeros_like(masks[0], dtype=np.float32)
    for idx in target_indices:
        combined_mask = np.maximum(combined_mask, masks[idx])

    mask = (combined_mask * 255).astype(np.uint8)
    mask = cv2.resize(mask, (img_rgb.shape[1], img_rgb.shape[0]))

    rgba = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2RGBA)
    rgba[:, :, 3] = mask

    info = {
        "pred_class": pred_class,
        "confidence": confidence,
        "num_instances": int(len(target_indices)),
        "all_pred_classes": [model_yolo.names[int(c)] for c in cls_ids],
        "all_confidences": [float(c) for c in confs],
    }

    return rgba, mask, result, info, "success"


## 6. YOLO出力保存関数

In [ ]:
def make_output_stem(image_path, true_class=None):
    """保存ファイル名のstemを作る。"""
    image_path = Path(image_path)
    prefix = true_class if true_class else "item"
    return f"{prefix}_{image_path.stem}"


def save_yolo_outputs(rgba, mask, yolo_result, image_path, true_class=None):
    """
    mask / transparent / annotated を保存し、保存パスdictを返す。
    """
    image_path = Path(image_path)
    stem = make_output_stem(image_path, true_class)

    mask_path = MASK_DIR / f"{stem}_mask.png"
    transparent_path = TRANSPARENT_DIR / f"{stem}_transparent.png"
    annotated_path = ANNOTATED_DIR / f"{stem}_annotated.png"

    Image.fromarray(mask).save(mask_path)
    Image.fromarray(rgba).save(transparent_path)

    annotated = yolo_result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    Image.fromarray(annotated_rgb).save(annotated_path)

    return {
        "mask_path": mask_path,
        "transparent_path": transparent_path,
        "annotated_path": annotated_path,
    }


## 7. OpenAI属性抽出関数

PoC考察を踏まえ、`dress` に「つなぎ・オーバーオール・ジャンプスーツ」を含めることをプロンプトで明示しています。

In [ ]:
def parse_json_safely(text):
    """OpenAIの出力をJSONとして安全に読み込む。"""
    try:
        data = json.loads(text)
    except Exception:
        cleaned = text.strip()
        cleaned = cleaned.replace("```json", "").replace("```", "").strip()

        try:
            data = json.loads(cleaned)
        except Exception:
            return {
                "category": None,
                "color_primary": None,
                "color_secondary": None,
                "pattern": None,
                "material": None,
                "silhouette": None,
                "raw_response": text,
                "openai_error": "json_parse_error",
            }

    default_keys = ["category", "color_primary", "color_secondary", "pattern", "material", "silhouette"]
    for key in default_keys:
        if key not in data:
            data[key] = None

    return data


def image_to_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def build_metadata_prompt():
    return """
このファッションアイテム画像を解析してください。

以下のJSON形式のみで返してください。

{
  "category": "",
  "color_primary": "",
  "color_secondary": "",
  "pattern": "",
  "material": "",
  "silhouette": ""
}

ルール:
- category は outer, tops, bottoms, dress, shoes, bag, hat, watch, glasses のいずれかに近いカテゴリで答える
- dress には ワンピース, ドレス, つなぎ, オーバーオール, ジャンプスーツ など上下が一体になった衣服を含める
- color_primary は主色を答える
- color_secondary は副色がなければ null
- pattern は柄・デザインを答える
- material は素材・質感を答える
- silhouette は形状・サイズ感・デザイン特徴を簡潔に答える
- 日本語で出力する
- JSON以外を出力しない
- 空欄は禁止
- 判断が難しい場合は「その他」を選ぶ

pattern は以下から最も近いものを必ず1つ選ぶ:
- 無地
- ストライプ
- ボーダー
- チェック
- ドット
- 花柄
- ロゴ
- プリント
- カモフラ
- その他

material は以下から最も近いものを必ず1つ選ぶ:
- コットン
- デニム
- ニット
- レザー
- ナイロン
- フリース
- ウール
- スウェット
- ファー
- ボア
- 金属
- 樹脂
- その他

注意:
- デニム、ニット、レザー、ナイロンなどは pattern ではなく material に入れる
- 無地のウィンドブレーカーは pattern=無地, material=ナイロン
- デニムパンツは pattern=無地, material=デニム
- ニット帽は pattern=無地, material=ニット
- ファーコートは pattern=無地, material=ファー
- ボアジャケットは pattern=無地, material=ボア
- 時計や眼鏡など素材判定が難しい場合は見た目から最も近い material を選ぶ
- 分からない場合は material="その他" とする
- material を空欄や null にしない
- pattern を空欄や null にしない

例:
{
  "category": "outer",
  "color_primary": "ブラウン",
  "color_secondary": null,
  "pattern": "無地",
  "material": "ファー",
  "silhouette": "ロングコート"
}
"""


def extract_metadata_with_openai(image_path, model_name=OPENAI_MODEL):
    """背景透過画像をOpenAI APIに渡し、属性JSONを得る。"""
    if client is None:
        return {
            "category": None,
            "color_primary": None,
            "color_secondary": None,
            "pattern": None,
            "material": None,
            "silhouette": None,
            "openai_error": "OPENAI_API_KEY_not_found",
        }

    prompt = build_metadata_prompt()
    base64_image = image_to_base64(image_path)

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                    },
                ],
            }
        ],
        response_format={"type": "json_schema"},
    )

    text = response.choices[0].message.content
    return parse_json_safely(text)


## 8. 1画像処理関数

FastAPI化するときは、まずこの `process_one_image()` を `pipeline_service.py` に移植すると楽です。

In [ ]:
def process_one_image(image_path, true_class=None, run_openai=True, conf=CONF_THRES):
    """
    1画像に対して SmartCloset AI のパイプラインを実行する。

    1. YOLOv8-segで対象物を切り出す
    2. mask / transparent / annotated を保存する
    3. 背景透過画像をOpenAIに渡して属性抽出する
    4. 結果を1行分のdictとして返す
    """
    image_path = Path(image_path)

    rgba, mask, yolo_result, yolo_info, status = segment_item(image_path, conf=conf)

    mask_path = None
    transparent_path = None
    annotated_path = None
    metadata = {}

    if status == "success":
        saved_paths = save_yolo_outputs(
            rgba=rgba,
            mask=mask,
            yolo_result=yolo_result,
            image_path=image_path,
            true_class=true_class,
        )

        mask_path = saved_paths["mask_path"]
        transparent_path = saved_paths["transparent_path"]
        annotated_path = saved_paths["annotated_path"]

        if run_openai:
            try:
                metadata = extract_metadata_with_openai(transparent_path)
            except Exception as e:
                metadata = {"openai_error": str(e)}

    return {
        "image_path": str(image_path),
        "file_name": image_path.name,
        "true_class": true_class,
        "yolo_status": status,
        "yolo_pred_class": yolo_info["pred_class"] if yolo_info else None,
        "yolo_confidence": yolo_info["confidence"] if yolo_info else None,
        "num_instances": yolo_info["num_instances"] if yolo_info else None,
        "all_pred_classes": json.dumps(yolo_info["all_pred_classes"], ensure_ascii=False) if yolo_info else None,
        "all_confidences": json.dumps(yolo_info["all_confidences"], ensure_ascii=False) if yolo_info else None,
        "mask_path": str(mask_path) if mask_path else None,
        "transparent_path": str(transparent_path) if transparent_path else None,
        "annotated_path": str(annotated_path) if annotated_path else None,
        "category": metadata.get("category"),
        "color_primary": metadata.get("color_primary"),
        "color_secondary": metadata.get("color_secondary"),
        "pattern": metadata.get("pattern"),
        "material": metadata.get("material"),
        "silhouette": metadata.get("silhouette"),
        "openai_error": metadata.get("openai_error"),
        "raw_response": metadata.get("raw_response"),
    }


## 9. 一括実行関数

In [ ]:
def run_pipeline(df_images, run_openai=True, save_csv=True, csv_path=RESULT_CSV_PATH, conf=CONF_THRES):
    """
    複数画像に対して `process_one_image()` を一括実行する。
    """
    results = []

    for i, row in df_images.iterrows():
        image_path = Path(row["image_path"])
        true_class = row.get("true_class", None)

        print(f"[{i + 1}/{len(df_images)}] {image_path}")

        result = process_one_image(
            image_path=image_path,
            true_class=true_class,
            run_openai=run_openai,
            conf=conf,
        )

        results.append(result)

    df_results = pd.DataFrame(results)

    if save_csv:
        csv_path = Path(csv_path)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df_results.to_csv(csv_path, index=False, encoding="utf-8-sig")
        print("saved:", csv_path)

    return df_results


## 10. 結果確認用の表示関数

In [ ]:
def _open_image_or_none(path):
    if path is None or path == "" or pd.isna(path):
        return None

    p = Path(path)
    if not p.exists():
        return None

    return Image.open(p)


def show_pipeline_result(row_or_df, idx=None):
    """
    pipeline結果を画像付きで確認する。

    show_pipeline_result(df_results, idx=0)
    show_pipeline_result(pd.Series(one_result))
    """
    if isinstance(row_or_df, pd.DataFrame):
        if idx is None:
            idx = 0
        row = row_or_df.iloc[idx]
    else:
        row = row_or_df

    original_img = _open_image_or_none(row.get("image_path"))
    mask_img = _open_image_or_none(row.get("mask_path"))
    transparent_img = _open_image_or_none(row.get("transparent_path"))
    annotated_img = _open_image_or_none(row.get("annotated_path"))

    print("=" * 60)
    print(f"file_name       : {row.get('file_name', '')}")
    print(f"true_class      : {row.get('true_class', '')}")
    print(f"yolo_status     : {row.get('yolo_status', '')}")
    print(f"yolo_pred_class : {row.get('yolo_pred_class', '')}")
    print(f"confidence      : {row.get('yolo_confidence', '')}")
    print("-" * 60)
    print("metadata")
    for col in ["category", "color_primary", "color_secondary", "pattern", "material", "silhouette", "openai_error"]:
        print(f"{col:16}: {row.get(col, '')}")

    imgs = [original_img, mask_img, transparent_img, annotated_img]
    titles = ["Original", "Mask", "Transparent", "Annotated"]

    plt.figure(figsize=(16, 4))
    for i, (img, title) in enumerate(zip(imgs, titles), start=1):
        plt.subplot(1, 4, i)
        if img is not None:
            if title == "Mask":
                plt.imshow(img, cmap="gray")
            else:
                plt.imshow(img)
        else:
            plt.text(0.5, 0.5, "None", ha="center", va="center")
        plt.title(title)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


## 11. 単発実行例

まずは `run_openai=False` でYOLOだけ確認するのがおすすめです。

In [ ]:
# 例1: input/ 直下の画像を使う場合
# df_images = collect_images_from_flat_dir(INPUT_DIR)

# 例2: PoCのclass別画像を使う場合
# df_images = collect_images_from_class_dirs(TEST_IMAGE_DIR)

# print("画像枚数:", len(df_images))
# df_images.head()


In [ ]:
# sample_path = df_images.iloc[0]["image_path"]
# sample_true_class = df_images.iloc[0].get("true_class", None)

# one_result = process_one_image(
#     image_path=sample_path,
#     true_class=sample_true_class,
#     run_openai=False,
# )

# one_result


In [ ]:
# show_pipeline_result(pd.Series(one_result))


## 12. 一括実行例

OpenAI APIを使う場合は `run_openai=True` にします。API料金やレート制限が気になる場合は、まず `False` でYOLO保存まで確認してください。

In [ ]:
# df_images = collect_images_from_class_dirs(TEST_IMAGE_DIR)
# print("画像枚数:", len(df_images))
# df_images.head()


In [ ]:
# df_results = run_pipeline(
#     df_images=df_images,
#     run_openai=True,
#     save_csv=True,
#     csv_path=RESULT_CSV_PATH,
#     conf=CONF_THRES,
# )

# df_results.head()


In [ ]:
# show_pipeline_result(df_results, idx=0)


## 13. FastAPI化するときの対応関係

| Notebook関数 | FastAPI移植先 |
|---|---|
| `segment_item()` | `services/yolo_service.py` |
| `save_yolo_outputs()` | `services/yolo_service.py` または `storage_service.py` |
| `extract_metadata_with_openai()` | `services/llm_service.py` |
| `process_one_image()` | `services/pipeline_service.py` |
| `run_pipeline()` | PoC/バッチ処理用。API本体では基本不要 |

APIでは最終的に以下のように使えます。

```python
result = process_one_image(uploaded_image_path, run_openai=True)
```
